# 01 — Exploratory data analysis

**Cilj:** razumeti šta imamo u sirovim podacima i koje probleme pipeline mora da reši.

Ovaj notebook pokriva strukturu podataka, duplikate, missing values, pozicije, minute, izbor početnih feature-a i distribuciju market value targeta. Transformacije i matching su namerno ostavljeni za naredne notebookove.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from src.data_processing import (
    ATTACKING_FEATURES,
    BASIC_FEATURES,
    DEFENSIVE_FEATURES,
    MISC_FEATURES,
    PASSING_FEATURES,
    POSSESSION_FEATURES,
)
from src.matching import prepare_market_values

players = pd.read_csv(RAW_DIR / "players_data-2024_2025.csv")
print(f"Rows: {len(players):,}; columns: {players.shape[1]}")
players.head()

## Duplikati i promene kluba

Isto ime može označavati transfer tokom sezone, ali i dve različite osobe. Zbog toga Player nije dovoljan identifikator.

In [ ]:
name_counts = players["Player"].value_counts()
duplicate_names = name_counts[name_counts > 1]
duplicate_rows = players.loc[players["Player"].isin(duplicate_names.index)]

print(f"Names appearing more than once: {len(duplicate_names):,}")
duplicate_rows[["Player", "Age", "Nation", "Squad", "Comp", "Pos", "Min"]].sort_values(
    ["Player", "Age", "Squad"]
).head(30)

## Missing values i pozicije

In [ ]:
missing = players.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_rows").head(30))
display(players["Pos"].value_counts().to_frame("rows"))

outfield = players.loc[players["Pos"] != "GK"].copy()
goalkeepers = players.loc[players["Pos"] == "GK"].copy()
print(f"Outfield rows: {len(outfield):,}; goalkeeper rows: {len(goalkeepers):,}")

## Minuti

Mali uzorci minuta su važni kasnije: deljenje ukupne statistike sa veoma malim brojem minuta proizvodi nestabilne per-90 vrednosti.

In [ ]:
display(outfield["Min"].describe())
thresholds = [90, 180, 450, 900]
pd.Series({f"Min < {limit}": int((outfield["Min"] < limit).sum()) for limit in thresholds})

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(outfield["Min"].dropna(), bins=30, edgecolor="white")
plt.xlabel("Minutes played")
plt.ylabel("Players")
plt.title("Outfield playing-time distribution")
plt.tight_layout()

## Početni feature set

Biramo interpretabilne napadačke, pas, possession, defanzivne i aerial metrike. Procenti se u agregaciji preračunavaju iz brojilaca i imenilaca, umesto da se prosečavaju između klubova.

In [ ]:
feature_groups = {
    "basic": BASIC_FEATURES,
    "attacking": ATTACKING_FEATURES,
    "passing": PASSING_FEATURES,
    "possession": POSSESSION_FEATURES,
    "defensive": DEFENSIVE_FEATURES,
    "misc": MISC_FEATURES,
}
pd.Series({name: len(columns) for name, columns in feature_groups.items()}, name="feature_count")

## Market-value target

Za sezonu 2024/25 uzimamo poslednju dostupnu valuaciju svakog igrača u intervalu 1. jul 2024 — 30. jun 2025.

In [ ]:
market_values, _ = prepare_market_values(
    RAW_DIR / "players.csv",
    RAW_DIR / "player_valuations.csv",
)
display(market_values["market_value_in_eur"].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(market_values["market_value_in_eur"].dropna() / 1_000_000, bins=50)
axes[0].set(title="Market value", xlabel="€ millions", ylabel="Players")
axes[1].hist(np.log1p(market_values["market_value_in_eur"].dropna()), bins=50)
axes[1].set(title="log1p(market value)", xlabel="log euros", ylabel="Players")
plt.tight_layout()

## Zaključci

- Goalkeepers zahtevaju poseban feature set i ovde ih isključujemo.
- Promene kluba zahtevaju agregaciju, ali same-name identiteti zahtevaju dodatnu proveru.
- Procentualne metrike mogu legitimno biti missing kada nema pokušaja.
- Target je izrazito right-skewed, pa modelujemo `log1p(market_value_in_eur)`.
- Mali broj minuta motiviše reliability shrinkage u notebooku 03.